# Lab 04 — TF-IDF

**Curso:** Diplomado en Inteligencia Artificial  
**Nivel:** Introductorio–Intermedio  
**Duración estimada:** 45–60 minutos  
**Prerequisito:** Lab 03 — Bag of Words

---

## Contexto — El problema de BoW

En el Lab anterior construiste una matriz BoW sobre el corpus de Wikipedia.  
Las palabras con mayor frecuencia total eran: *de, la, el, en, que, se, un…*

Estas palabras aparecen en **todos** los documentos. BoW les asigna conteos altísimos  
y eso hace que dominen cualquier comparación. Dos artículos sobre temas distintos  
parecen similares porque comparten 
, 
 y 
.

**La pregunta clave:**
> ¿Cómo diferenciamos una palabra informativa (como `"Pirineos"`) de una que no aporta nada (`"de"`)?

La respuesta es **TF-IDF**.

> 📌 TF-IDF no es una alternativa a BoW — es una **mejora**.  
> Sigue siendo una bolsa de palabras, pero los conteos se reemplazan por pesos  
> que reflejan la importancia real de cada término en cada documento.

In [ ]:
# ⚙️ Configuración — Ejecuta esta celda primero
!pip install scikit-learn pandas --quiet

import os
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("✅ Librerías cargadas correctamente")

---

## ⚙️ Carga del Corpus

Usaremos el mismo `eswiki_corpus.txt` del Lab anterior.

> ⚠️ **Si estás en Google Colab:** ejecuta la siguiente celda para subir el archivo o montarlo desde Drive.  
> Si estás en local (VS Code / Jupyter), puedes saltar esa celda — el archivo se lee directamente.

In [ ]:
import os

EN_COLAB = 'google.colab' in str(get_ipython())

if EN_COLAB:
    # --- OPCIÓN A1: Montar Google Drive ---
    # Sube eswiki_corpus.txt a tu Drive (ej. en /Mi unidad/PLN/)
    # y ajusta la ruta CORPUS_FILE abajo.

    # from google.colab import drive
    # drive.mount('/content/drive')
    # CORPUS_FILE = '/content/drive/MyDrive/PLN/eswiki_corpus.txt'

    # --- OPCIÓN A2: Subir el archivo directamente ---
    from google.colab import files
    print("Sube el archivo eswiki_corpus.txt cuando aparezca el selector:")
    uploaded = files.upload()
    CORPUS_FILE = list(uploaded.keys())[0]

else:
    CORPUS_FILE = os.path.join(os.path.dirname(os.path.abspath('__file__')),
                               '..', 'eswiki_corpus.txt')

print(f"✅ Ruta del corpus: {CORPUS_FILE}")
print(f"   Existe: {os.path.exists(CORPUS_FILE)}")

In [ ]:
# ⚙️ Función de limpieza + carga del corpus
# Ejecuta esta celda antes de continuar

def limpiar(texto):
    """Elimina ruido típico de un dump de Wikipedia en español."""
    texto = re.sub(r'&lt;.*?&gt;', ' ', texto)
    texto = re.sub(r'<[^>]+>', ' ', texto)
    texto = re.sub(r'\[\[.*?\]\]', ' ', texto)
    texto = re.sub(r'\{\{.*?\}\}', ' ', texto)
    texto = re.sub(r"'{2,}", '', texto)
    texto = re.sub(r'&\w+;', ' ', texto)
    texto = re.sub(r'[^a-záéíóúüñA-ZÁÉÍÓÚÜÑ\s]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip().lower()
    return texto

STOPWORDS_ES = [
    'a', 'al', 'algo', 'algunas', 'algunos', 'ante', 'antes', 'como',
    'con', 'contra', 'cual', 'cuando', 'de', 'del', 'desde', 'donde',
    'durante', 'e', 'el', 'ella', 'ellas', 'ellos', 'en', 'entre',
    'era', 'es', 'esa', 'esas', 'ese', 'eso', 'esos', 'esta', 'estas',
    'este', 'estos', 'fue', 'ha', 'hay', 'he', 'la', 'las', 'le', 'les',
    'lo', 'los', 'mas', 'me', 'mi', 'muy', 'ni', 'no', 'nos', 'o', 'os',
    'para', 'pero', 'por', 'que', 'se', 'si', 'sin', 'sobre', 'son',
    'su', 'sus', 'tambien', 'tanto', 'te', 'tiene', 'tu', 'tus',
    'un', 'una', 'uno', 'unos', 'unas', 'y', 'ya', 'yo', 'ser', 'han',
    'sido', 'parte', 'dicho', 'tras',
]

N_LINEAS = 50   # Cambia este número para usar más oraciones

with open(CORPUS_FILE, 'r', encoding='utf-8') as f:
    lineas_raw = [l.strip() for l in f if l.strip()]

CORPUS_RAW = lineas_raw[:N_LINEAS]
oraciones_limpias = [limpiar(o) for o in CORPUS_RAW]
oraciones_limpias = [o for o in oraciones_limpias if len(o.split()) > 3]

print(f"✅ Corpus cargado: {len(CORPUS_RAW)} líneas leídas → {len(oraciones_limpias)} oraciones válidas")
print(f"\nEjemplo:")
print(f"  ANTES:   {CORPUS_RAW[2][:120]}")
print(f"  DESPUÉS: {oraciones_limpias[2][:120]}")

---

## Parte 1 — Cómo funciona TF-IDF (demostración)

> ⚠️ **Esta sección usa un mini-corpus inventado de 3 oraciones.**  
> Su único propósito es entender la matemática con números pequeños.  
> El corpus real de Wikipedia que usarás en las Partes 2 y 3 tiene contenido muy diferente — y eso está bien.

Mini-corpus de ejemplo:

| Doc | Texto |
|-----|-------|
| D1  | `el gato duerme en el sofá` |
| D2  | `el perro duerme en el jardín` |
| D3  | `el gato persigue el perro` |

---

### Paso 1 — TF (Term Frequency)

$$TF(t, d) = \frac{\text{veces que aparece } t \text{ en } d}{\text{total de palabras en } d}$$

TF para **D1** (6 palabras en total):

| Palabra | Conteo en D1 | TF = conteo / 6 |
|---------|--------------|-----------------|
| el      | 2            | 2/6 = **0.33**  |
| gato    | 1            | 1/6 = **0.17**  |
| duerme  | 1            | 1/6 = **0.17**  |
| en      | 1            | 1/6 = **0.17**  |
| sofá    | 1            | 1/6 = **0.17**  |

> Nota: `"el"` aparece 2 veces → su TF es el doble. Pero eso no significa que sea más "informativa".

---

### Paso 2 — IDF (Inverse Document Frequency)

$$IDF(t) = \ln\left(\frac{N}{df_t}\right)$$

Donde **N** = total de documentos y **df_t** = documentos que contienen el término.

> Si una palabra aparece en **todos** los documentos → IDF = 0 (no distingue nada).  
> Si aparece en **solo uno** → IDF máximo (muy específica).

Tabla IDF con **N = 3** documentos:

| Palabra | ¿En D1? | ¿En D2? | ¿En D3? | df | IDF = ln(3/df) |
|---------|---------|---------|---------|----|--------------------|
| el      | Sí      | Sí      | Sí      | 3  | ln(3/3) = **0.00** |
| gato    | Sí      | No      | Sí      | 2  | ln(3/2) = **0.41** |
| duerme  | Sí      | Sí      | No      | 2  | ln(3/2) = **0.41** |
| en      | Sí      | Sí      | No      | 2  | ln(3/2) = **0.41** |
| jardín  | No      | Sí      | No      | 1  | ln(3/1) = **1.10** |
| perro   | No      | Sí      | Sí      | 2  | ln(3/2) = **0.41** |
| sofá    | Sí      | No      | No      | 1  | ln(3/1) = **1.10** |

> 💡 `"el"` tiene IDF = 0 porque aparece en **los 3 documentos** — no aporta información para distinguirlos.  
> `"sofá"` tiene IDF = 1.10 porque solo aparece en D1 — es muy característica de ese documento.

---

### Paso 3 — TF-IDF = TF × IDF

TF-IDF final para **D1**:

| Palabra | TF   | IDF  | TF-IDF = TF × IDF |
|---------|------|------|-------------------|
| el      | 0.33 | 0.00 | **0.00**          |
| gato    | 0.17 | 0.41 | **0.07**          |
| duerme  | 0.17 | 0.41 | **0.07**          |
| en      | 0.17 | 0.41 | **0.07**          |
| sofá    | 0.17 | 1.10 | **0.19**          |

> 💡 La palabra más característica de D1 es **`"sofá"`** — aunque solo aparece 1 vez, no aparece en ningún otro documento.  
> `"el"` aparece 2 veces pero su peso es **0** porque no distingue nada.

---

**Esto es exactamente lo que hace `TfidfVectorizer` de sklearn — automáticamente, para todos los documentos.**  
En el siguiente ejercicio lo verificamos con código.

In [ ]:
# Ejercicio 1 — Verifica tus cálculos manuales con sklearn
# Descomenta y ejecuta

# mini_corpus = [
#     "el gato duerme en el sofa",
#     "el perro duerme en el jardin",
#     "el gato persigue el perro",
# ]

# vectorizador_mini = TfidfVectorizer(use_idf=True, smooth_idf=False, norm=None)
# matriz_mini       = vectorizador_mini.fit_transform(mini_corpus)
# tabla_mini        = pd.DataFrame(
#     matriz_mini.toarray().round(3),
#     columns=vectorizador_mini.get_feature_names_out(),
#     index=["D1", "D2", "D3"]
# )
# print("Matriz TF-IDF del mini-corpus:")
# print(tabla_mini)
# print()
# print("⚠️  Los números NO coinciden exactamente con el cálculo manual por dos razones:")
# print()
# print("  1) TF: el manual usa TF = conteo / total_palabras (frecuencia relativa)")
# print("         sklearn usa TF = conteo crudo  (sin dividir por la longitud del documento)")
# print("         → Por eso 'el' tiene TF=2 en sklearn, no TF=0.33")
# print()
# print("  2) IDF: el manual usa IDF = ln(N/df)         → 'el' da 0.00")
# print("          sklearn usa IDF = ln(N/df) + 1        → 'el' da 1.00  (nunca llega a 0)")
# print("          El +1 es un ajuste técnico para que ninguna palabra quede completamente ignorada.")
# print()
# print("  ✅ Lo importante: el RANKING es el mismo.")
# print("     La palabra EXCLUSIVA de cada documento sigue siendo la más característica.")
# print()
# print("Palabra más característica por documento:")
# for i, doc in enumerate(["D1", "D2", "D3"]):
#     fila = tabla_mini.iloc[i]
#     print(f"  {doc}: '{fila.idxmax()}' (TF-IDF = {fila.max():.3f})")

---

## Parte 2 — `TfidfVectorizer` sobre el corpus real

`TfidfVectorizer` aplica automáticamente TF-IDF a todos los documentos del corpus.

### 📝 Ejercicio 2a — Aplicar TF-IDF al corpus

Descomenta y ejecuta.

> ¿Los valores de la matriz son enteros o decimales? ¿Por qué?  
> ¿Cuántas palabras distintas (columnas) captura con `max_features=500`?

In [ ]:
# Ejercicio 2a — Descomenta y ejecuta

# vectorizador  = TfidfVectorizer(max_features=500, stop_words=STOPWORDS_ES)
# matriz_tfidf  = vectorizador.fit_transform(oraciones_limpias)

# print(f"Forma de la matriz TF-IDF: {matriz_tfidf.shape}")
# print(f"  → {matriz_tfidf.shape[0]} documentos  ×  {matriz_tfidf.shape[1]} palabras")

### 📝 Ejercicio 2b — Palabras más características por documento

Descomenta y ejecuta. Muestra las top-5 palabras con mayor TF-IDF para los primeros 3 documentos.

> ¿Las palabras resultantes describen bien el tema del documento?  
> Compáralas con las palabras que obtendrías con BoW puro (conteos) — ¿cuál es más informativa?

In [ ]:
# Ejercicio 2b — Descomenta y ejecuta (requiere haber ejecutado 2a)

# tabla_tfidf = pd.DataFrame(
#     matriz_tfidf.toarray(),
#     columns=vectorizador.get_feature_names_out()
# )

# print("Top-5 palabras más características por documento (TF-IDF):")
# print("-" * 60)
# for i in range(min(3, len(tabla_tfidf))):
#     fila    = tabla_tfidf.iloc[i]
#     top5    = fila.nlargest(5)
#     reporte = "  ,  ".join([f"{palabra} ({peso:.3f})" for palabra, peso in top5.items()])
#     print(f"  Doc {i}: {reporte}")

---

## Parte 3 — Similitud Coseno

TF-IDF convierte cada documento en un vector. Podemos medir qué tan similares son  
dos documentos calculando el **ángulo** entre sus vectores:

$$\text{similitud}(d_1, d_2) = \cos(\theta) = \frac{d_1 \cdot d_2}{||d_1|| \cdot ||d_2||}$$

El resultado está entre **0** (completamente distintos) y **1** (idénticos).

### 📝 Ejercicio 3a — Predicción con el mini-corpus

> ⚠️ Esta predicción es sobre el **mini-corpus de 3 oraciones** de la Parte 1.  
> El corpus real (Wikipedia) puede dar resultados muy distintos.

Con el mini-corpus:

| Par | Palabras compartidas (IDF > 0) | ¿Cuál esperas más similar? |
|-----|-------------------------------|----------------------------|
| D1 – D2 | `"duerme"` (df=2), `"en"` (df=2) | |
| D1 – D3 | `"gato"` (df=2) | |
| D2 – D3 | `"perro"` (df=2) | |

> ¿Qué par de documentos esperas que tenga **mayor** similitud coseno? ¿Por qué?

*(Escribe tu predicción aquí antes de ejecutar)*

---

### 📝 Ejercicio 3b — Similitud sobre el corpus real

Descomenta y ejecuta. Se muestra la matriz de similitud para los primeros 10 documentos.

> ¿Qué par tiene mayor similitud? ¿Tiene sentido temáticamente?  

> ¿Qué par tiene similitud 0.0? ¿Por qué?

In [ ]:
# Ejercicio 3 — Descomenta y ejecuta (requiere haber ejecutado 2a)

# N_DOCUMENTOS     = min(10, len(oraciones_limpias))
# similitudes      = cosine_similarity(matriz_tfidf[:N_DOCUMENTOS])
# tabla_similitud  = pd.DataFrame(
#     similitudes.round(2),
#     columns=[f"D{i}" for i in range(N_DOCUMENTOS)],
#     index=[f"D{i}" for i in range(N_DOCUMENTOS)],
# )

# print(f"Matriz de similitud coseno (primeros {N_DOCUMENTOS} documentos):")
# print(tabla_similitud)
# print()

# # Par más similar (ignora la diagonal)
# copia_sim = similitudes.copy()
# np.fill_diagonal(copia_sim, 0)
# idx_max   = copia_sim.argmax()
# i, j      = divmod(idx_max, N_DOCUMENTOS)
# print(f"Par más similar: D{i} y D{j}  (similitud = {copia_sim[i,j]:.3f})")
# print(f"  D{i}: {oraciones_limpias[i][:80]}")
# print(f"  D{j}: {oraciones_limpias[j][:80]}")

---

## Parte 4 (Bonus) — TF-IDF con N-gramas

### 📝 Ejercicio 4 — Bigramas + TF-IDF

Descomenta y ejecuta con `ngram_range=(1, 2)` (unigramas y bigramas).

> ¿Aparecen bigramas entre las palabras más características?  

> ¿Mejora la representación al combinar unigramas y bigramas?

In [ ]:
# Ejercicio 4 (Bonus) — Descomenta y ejecuta

# vectorizador_bigramas = TfidfVectorizer(ngram_range=(1, 2), max_features=500,
#                                         stop_words=STOPWORDS_ES)
# matriz_bigramas       = vectorizador_bigramas.fit_transform(oraciones_limpias)
# tabla_bigramas        = pd.DataFrame(
#     matriz_bigramas.toarray(),
#     columns=vectorizador_bigramas.get_feature_names_out()
# )

# print("Top-5 términos más característicos por documento (unigramas + bigramas):")
# print("-" * 60)
# for i in range(min(3, len(tabla_bigramas))):
#     fila    = tabla_bigramas.iloc[i]
#     top5    = fila.nlargest(5)
#     reporte = "  ,  ".join([f"{termino} ({peso:.3f})" for termino, peso in top5.items()])
#     print(f"  Doc {i}: {reporte}")
# print()
# print("→ ¿Ves bigramas en el top-5?")

---

## 🤔 Reflexión Final

Responde en las celdas de texto a continuación (doble clic para editar).

---

**1. IDF = 0**

Si una palabra aparece en **todos** los documentos del corpus, su IDF es 0 y su TF-IDF también.  
¿Qué tipo de palabras caen en esta categoría?  
¿Cuándo podría esto ser un problema?

*(Escribe tu respuesta aquí)*

---

**2. Tamaño del corpus importa**

El IDF de `"gato"` en el mini-corpus de 3 documentos es muy distinto al de un corpus de 50 000.  
¿Por qué? ¿Mejora o empeora la representación al aumentar el corpus?

*(Escribe tu respuesta aquí)*

---

**3. TF-IDF vs Word2Vec**

TF-IDF sigue siendo una bolsa de palabras.  
`"rey"` y `"monarca"` tienen vectores completamente independientes aunque signifiquen lo mismo.  
¿Qué representación necesitarías para que palabras con significados similares  
produzcan vectores similares?

*(Escribe tu respuesta aquí)*

